<a href="https://colab.research.google.com/github/BumRushBangz/Summer26Projects/blob/main/Ch4FromScratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
#create practice table
y_true = [1,1,0,0,1]
y_pred = [1,0,0,0,1]

#create a confusion matrix operation, adds to the count of tp,fp,tn, and fn as they appear.
def confusion_matrix(y_true,y_pred):
  tp=0; fp=0; tn=0;fn=0
  for i in range(len(y_true)):
        if y_true[i] == 1 and y_pred[i] == 1:
            tp = tp + 1
        if y_true[i] == 1 and y_pred[i] == 0:
            fn = fn + 1
        if y_true[i] == 0 and y_pred[i] == 0:
            tn = tn + 1
        if y_true[i] == 0 and y_pred[i] == 1:
            fp = fp + 1
  return tp, fp, tn, fn

practice = confusion_matrix(y_true,y_pred)
print(practice)

#create sensitivity and specificity functions
def sensitivity(tp, fn):
    return tp / (tp + fn)

def specificity(tn, fp):
    return tn / (tn + fp)

#create a summary like statsmodels has
def classification_report(y_true, y_pred):
    tp, fp, tn, fn = confusion_matrix(y_true, y_pred)
    print("TP:", tp, " FP:", fp)
    print("TN:", tn, " FN:", fn)
    print("Sensitivity:", sensitivity(tp, fn))
    print("Specificity:", specificity(tn, fp))
    print("Accuracy:   ", (tp + tn) / len(y_true))

classification_report(y_true,y_pred)

(2, 0, 2, 1)
TP: 2  FP: 0
TN: 2  FN: 1
Sensitivity: 0.6666666666666666
Specificity: 1.0
Accuracy:    0.8


In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/lazappi/nba_positions/master/nba_positions.tsv"
df = pd.read_csv(url, sep="\t")     # sep="\t" because it's tab-separated

print(df.head())
print(df['Position'].value_counts())

  Position  TurnoverPct  ReboundPct  AssistPct  FieldGoalPct
0   Center         10.6        17.4       14.7          50.2
1   Center          9.6        19.6       16.9          46.8
2   Center         16.0        14.2        5.4          57.1
3   Center         17.2        14.9        2.7          58.3
4   Center         11.7        20.8        4.4          55.7
Position
Center           50
PointGuard       50
ShootingGuard    50
Name: count, dtype: int64


In [12]:
# Binary target: guard = 1, center = 0
y = [1 if pos != "Center" else 0 for pos in df['Position']]

# Features (the four stats)
rebound = df['ReboundPct'].tolist()
assist  = df['AssistPct'].tolist()
fg = df['FieldGoalPct'].tolist()
TO = df['TurnoverPct'].tolist()

def standardize(values):
    n = len(values)
    mean = sum(values) / n
    # Standard deviation
    var = 0
    for v in values:
        var = var + (v - mean)**2
    std = (var / n)**0.5
    # Rescale each value to std devs from mean.
    scaled = []
    for v in values:
        scaled.append((v - mean) / std)
    return scaled

rebound_s = standardize(rebound)
assist_s = standardize(assist)
fg_s = standardize(fg)
TO_s = standardize(TO)

# Each player = one row of their stats  [rebound, assist, fg, TO]
players = []
for i in range(len(y)):
    players.append([rebound_s[i], assist_s[i], fg_s[i], TO_s[i]])

def distance(p1, p2):
    total = 0
    for k in range(len(p1)):
        total = total + (p1[k] - p2[k])**2
    return total**0.5
# This just cycles through each stat for each player, using the
# pythagorean theorem to assess distance.

def k_nearest_labels(players, labels, target, k):
    dist = []
    # measure distance from target to every player, remember each one's label,
    # find K nearest.
    for i in range(len(players)):
        d = distance(target, players[i])
        dist.append([d, labels[i]])
        # pair the distance with that player's label in dist.

    # Sort by distance (smallest first).
    dist.sort()

    # Take the labels of the k closest.
    nearest = []
    for i in range(k):
        nearest.append(dist[i][1])
        # [1] grabs the label from each [dist, label] pairing.
    return nearest

def knn_predict(players, labels, target, k):
    nearest = k_nearest_labels(players, labels, target, k)
    return sum(nearest)/k

# Could simplify like this:
def predict(target, k):
    return knn_predict(players, y, target, k)

In [15]:
# Now we can predict probability of guard for every player.
probs = []
for i in range(len(players)):
    probs.append(knn_predict(players, y, players[i], 20))

print(probs[:10])

[0.0, 0.05, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0]


In [20]:
# ROC: sweep the threshold, record (sensitivity, specificity) at each.
roc_points = []

# Try thresholds from 0.0 up to 1.0 in steps of 0.05
threshold = 0.0
while threshold <= 1.0:
    # Step 1: turn probabilities into 0/1 predictions at each threshold,
    # starting here.
    preds = []
    for p in probs:
        if p >= threshold:
            preds.append(1)      # predict guard
        else:
            preds.append(0)      # predict center

    # Step 2: score those predictions.
    tp, fp, tn, fn = confusion_matrix(y, preds)
    sens = sensitivity(tp, fn)
    spec = specificity(tn, fp)

    # Step 3: record the point.
    roc_points.append([threshold, sens, spec])

    # Iterate
    threshold = threshold + 0.05

In [23]:
# Turn roc_points into (x, y) = (1 - specificity, sensitivity), sorted by x.
xy = []
for point in roc_points:
    threshold, sens, spec = point
    xy.append([1 - spec, sens])
xy.sort()      # Order by x (1 - specificity), left to right.

auc = 0
for i in range(1, len(xy)):
    x_prev, y_prev = xy[i-1]
    x_curr, y_curr = xy[i]
    width  = x_curr - x_prev           # how wide this slice is
    avg_height = (y_curr + y_prev) / 2  # average of the two heights
    auc = auc + width * avg_height      # slice area, added to the total

print("AUC:", auc)

from sklearn.metrics import roc_auc_score
print("sklearn AUC:", roc_auc_score(y, probs))

AUC: 1.0
sklearn AUC: 1.0
